# 🦈 Marine Incident Fatality Predictor — Unified Dataset Dashboard
**Cloud ML Project — Interactive Ensemble Prototype (LO4: Cloud-based end-user service)**

This notebook provides a **Gradio web interface** that:
- Accepts user inputs (year, month, age, gender, activity type, incident type, location)
- Returns **ensemble predictions from all 3 trained models** (Random Forest, SVM, XGBoost)
- Displays **visual comparison** with bar chart showing each model's confidence
- Shows **consensus verdict** (ensemble average fatality probability)
- Performs **batch inference** on test set with comprehensive performance metrics

## Dataset Approach
✅ **Unified Dataset** — All 3 models trained on the same combined data (34,000+ records, 80/20 stratified split)  
✅ **Fair Comparison** — Same train/test split across all algorithms  
✅ **9 Features** — year, month, age, gender, activity_enc, inc_type_enc, latitude, longitude, source  
✅ **Binary Classification** — Predicts fatal (1) vs non-fatal (0) marine incidents

> **Run order:** Execute `data_preparation.ipynb` first, then the 3 training notebooks (teammate1_random_forest.ipynb, teammate2_svm.ipynb, teammate3_xgboost.ipynb), then run this dashboard.

## 1. Install & Import Dependencies

In [ ]:
import subprocess, sys
for pkg in ['gradio', 'folium']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import os, joblib, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import gradio as gr
import folium
from folium.plugins import MarkerCluster

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

print(f'Gradio  version: {gr.__version__}')
print(f'Folium  version: {folium.__version__}')

## 2. Colab / Cloud Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = '/content/drive/MyDrive/CloudML/'
else:
    BASE_DIR = './'

MODEL_DIR = os.path.join(BASE_DIR, 'models')
DATA_DIR  = BASE_DIR
print(f'Base dir   : {BASE_DIR}')
print(f'Model dir  : {MODEL_DIR}')

## 3. Load Trained Models & Reference Data

In [ ]:
# Load 3 trained combined models (unified dataset approach)
models = {}
for algo, prefix in [('RF', 'random_forest'), ('SVM', 'svm'), ('XGB', 'xgboost')]:
    path = os.path.join(MODEL_DIR, f'{prefix}_combined.pkl')
    if os.path.exists(path):
        models[algo] = joblib.load(path)
        print(f'✓ Loaded {prefix}_combined.pkl')
    else:
        print(f'✗ MISSING: {path}  (run training notebooks first)')

print(f'\nTotal models loaded: {len(models)}/3')

# Load combined dataset for reference statistics
combined_path = os.path.join(DATA_DIR, 'combined_dataset.csv')
if os.path.exists(combined_path):
    combined_df = pd.read_csv(combined_path)
    print(f'Combined dataset loaded: {len(combined_df)} records')
else:
    print(f'✗ Combined dataset not found')

# Load train/test sets (for dataset summary + test set inference)
train_path = os.path.join(DATA_DIR, 'combined_train.csv')
if os.path.exists(train_path):
    train_df = pd.read_csv(train_path)
    print(f'Train dataset loaded: {len(train_df)} records')
else:
    print(f'✗ Train dataset not found')

test_path = os.path.join(DATA_DIR, 'combined_test.csv')
if os.path.exists(test_path):
    test_df = pd.read_csv(test_path)
    print(f'Test dataset loaded: {len(test_df)} records')
else:
    print(f'✗ Test dataset not found')

# Load activity and incident type encodings for reference
try:
    activity_encoding_df = pd.read_csv(os.path.join(DATA_DIR, 'activity_encoding.csv'))
    inc_type_encoding_df = pd.read_csv(os.path.join(DATA_DIR, 'inc_type_encoding.csv'))
    print(f'Encodings loaded: {len(activity_encoding_df)} activities, {len(inc_type_encoding_df)} incident types')
except:
    print('Encoding files not found (optional)')

## 4. Build Label Encoders & Prediction Logic

In [ ]:
# Load human-readable label mappings (produced by data_preparation.ipynb)
try:
    act_map_path = os.path.join(DATA_DIR, 'activity_encoding.csv')
    inc_map_path = os.path.join(DATA_DIR, 'inc_type_encoding.csv')
    act_map_df = pd.read_csv(act_map_path)
    inc_map_df = pd.read_csv(inc_map_path)

    # label -> code and code -> label lookups
    ACTIVITY_LABEL_TO_CODE = dict(zip(act_map_df['activity'], act_map_df['encoded']))
    INC_TYPE_LABEL_TO_CODE = dict(zip(inc_map_df['inc_type'], inc_map_df['encoded']))

    ACTIVITY_CHOICES = sorted(ACTIVITY_LABEL_TO_CODE.keys())
    INC_TYPE_CHOICES = sorted(INC_TYPE_LABEL_TO_CODE.keys())
    print(f'✓ Loaded {len(ACTIVITY_CHOICES)} activity labels, {len(INC_TYPE_CHOICES)} incident type labels')
except Exception as e:
    # Fallback choices if encoding files not found
    ACTIVITY_LABEL_TO_CODE = {'Surfing': 0, 'Swimming': 1, 'Diving': 2, 'Fishing': 3, 'Unknown': 4}
    INC_TYPE_LABEL_TO_CODE = {'Unprovoked': 0, 'Provoked': 1, 'Unknown': 2}
    ACTIVITY_CHOICES = sorted(ACTIVITY_LABEL_TO_CODE.keys())
    INC_TYPE_CHOICES = sorted(INC_TYPE_LABEL_TO_CODE.keys())
    print(f'⚠ Encoding files not found, using fallback labels ({e})')

GENDER_LABELS = {'M': 0, 'F': 1, 'Unknown': -1}

# Since the dataset is unified (no per-dataset selection needed), we derive
# a single combined median location and the most common 'source' value
# automatically — end users never see or choose a source dataset.
if 'combined_df' in locals() and {'latitude', 'longitude'}.issubset(combined_df.columns):
    DEFAULT_LAT = float(combined_df['latitude'].median())
    DEFAULT_LON = float(combined_df['longitude'].median())
else:
    DEFAULT_LAT, DEFAULT_LON = 0.0, 0.0

if 'combined_df' in locals() and 'source' in combined_df.columns:
    DEFAULT_SOURCE = int(combined_df['source'].mode().iloc[0])
else:
    DEFAULT_SOURCE = 0

# Feature column order (must match training data)
FEATURE_COLS = ['year', 'month', 'age', 'gender', 'activity_enc', 'inc_type_enc',
                'latitude', 'longitude', 'source']
TARGET_COL = 'fatal'

def build_feature_row(year, month, age, gender_label, activity_label, inc_type_label):
    """
    Build the 9-feature vector for the unified combined dataset.
    Latitude/longitude/source are auto-filled from the combined dataset's
    overall statistics — end users only pick meaningful, labelled inputs.

    Args:
        year: year (int)
        month: month 1-12 (int)
        age: age in years (float)
        gender_label: 'M' / 'F' / 'Unknown'
        activity_label: human-readable activity (str)
        inc_type_label: human-readable incident type (str)
    """
    gender_code   = GENDER_LABELS.get(gender_label, -1)
    activity_code = ACTIVITY_LABEL_TO_CODE.get(activity_label, 0)
    inc_type_code = INC_TYPE_LABEL_TO_CODE.get(inc_type_label, 0)

    return np.array([[
        float(year),
        float(month),
        float(age) if age else 30.0,
        float(gender_code),
        float(activity_code),
        float(inc_type_code),
        DEFAULT_LAT,
        DEFAULT_LON,
        float(DEFAULT_SOURCE)
    ]])

print('✓ Encoders & feature builder ready (combined dataset)')
print(f'Feature columns: {FEATURE_COLS}')
print(f'Predicting target: {TARGET_COL}')
print(f'Default location (combined median): lat={DEFAULT_LAT:.2f}, lon={DEFAULT_LON:.2f}')

## 5. Map Builder

In [ ]:
def build_map(fatal_prob: float) -> str:
    """
    Return an HTML string of a Folium map showing a sample of historical
    incidents from the combined dataset, plus a ring marking the current
    ensemble prediction's fatal probability at the overall median location.
    Red markers = historical fatal, Blue = historical non-fatal.
    """
    if 'combined_df' in globals() and {'latitude', 'longitude', 'fatal'}.issubset(combined_df.columns):
        subset = combined_df[combined_df['latitude'].notna() &
                             combined_df['longitude'].notna()].copy()
    else:
        subset = pd.DataFrame(columns=['latitude', 'longitude', 'fatal'])

    centre_lat = subset['latitude'].median()  if len(subset) else 0.0
    centre_lon = subset['longitude'].median() if len(subset) else 0.0
    if pd.isna(centre_lat): centre_lat = 0.0
    if pd.isna(centre_lon): centre_lon = 0.0

    zoom = 2
    m = folium.Map(location=[centre_lat, centre_lon], zoom_start=zoom,
                   tiles='CartoDB positron')

    fatal_cluster  = MarkerCluster(name='Fatal (historical)').add_to(m)
    rescue_cluster = MarkerCluster(name='Non-Fatal (historical)').add_to(m)

    # Cap markers for performance in the demo
    for _, row in subset.sample(n=min(len(subset), 500), random_state=42).iterrows():
        is_fatal = row['fatal'] == 1
        icon   = folium.Icon(color='red' if is_fatal else 'blue', icon='info-sign')
        tip    = (f"Fatal: {'Yes' if is_fatal else 'No'}<br>"
                  f"Lat/Lon: {row['latitude']:.2f}, {row['longitude']:.2f}")
        cluster = fatal_cluster if is_fatal else rescue_cluster
        folium.Marker(location=[row['latitude'], row['longitude']],
                      popup=folium.Popup(tip, max_width=200),
                      icon=icon).add_to(cluster)

    # Prediction ring on centre
    ring_color = '#c9533e' if fatal_prob >= 0.5 else '#495b8d'
    folium.Circle(location=[centre_lat, centre_lon],
                  radius=80000,
                  color=ring_color, fill=True, fill_opacity=0.12,
                  popup=f'Ensemble prediction: {fatal_prob*100:.1f}% fatal probability'
                  ).add_to(m)

    folium.LayerControl().add_to(m)
    return m._repr_html_()

print('Map builder ready (uses combined_df, full dataset sample).')

## 6. Prediction Function

In [ ]:
def predict_and_visualise(year, month, age, gender_label, activity_label, inc_type_label):
    """
    Run inference on all 3 combined models and return visual comparisons.
    Latitude/longitude/source are resolved internally from the combined
    dataset's overall statistics, so end users only ever provide meaningful,
    labelled inputs (no dataset selection needed).
    
    Returns:
      - predictions_df: DataFrame with all 3 model predictions
      - comparison_chart: HTML bar chart comparing model probabilities
      - map_html: HTML historical incident map with prediction overlay
      - verdict_text: Summary verdict
    """
    # Build feature vector
    features = build_feature_row(year, month, age, gender_label, activity_label,
                                 inc_type_label)
    
    results = []
    probabilities = []
    predictions = []
    
    for algo_name in ['RF', 'SVM', 'XGB']:
        if algo_name in models:
            try:
                model = models[algo_name]
                prob = float(model.predict_proba(features)[0, 1])
                pred = 1 if prob >= 0.5 else 0
                pred_label = 'FATAL ⚠️' if prob >= 0.5 else 'Non-Fatal ✓'
                confidence = 'High' if abs(prob - 0.5) > 0.2 else 'Medium'
                
                results.append({
                    'Model': algo_name,
                    'Prediction': pred_label,
                    'Fatal Probability': f'{prob*100:.2f}%',
                    'Confidence': confidence,
                })
                probabilities.append(prob)
                predictions.append(pred)
            except Exception as e:
                results.append({
                    'Model': algo_name,
                    'Prediction': 'Error',
                    'Fatal Probability': f'Error: {str(e)[:20]}',
                    'Confidence': '—',
                })
        else:
            results.append({
                'Model': algo_name,
                'Prediction': 'Not Loaded',
                'Fatal Probability': '—',
                'Confidence': '—',
            })
    
    predictions_df = pd.DataFrame(results)
    
    # Calculate ensemble metrics
    n_models = len(probabilities)
    if n_models > 0:
        avg_prob = np.mean(probabilities)
        consensus = sum(1 for p in predictions if p == 1)
        consensus_label = f'{consensus}/{n_models} models predict FATAL'
    else:
        avg_prob = 0.5
        consensus_label = 'No models loaded'
    
    # Create visual comparison bar chart (HTML)
    if n_models > 0:
        chart_html = f"""
        <div style="margin: 20px 0; padding: 15px; background: #f8f9fa; border-radius: 8px;">
            <h4 style="margin-top: 0;">Model Probability Comparison</h4>
            <div style="display: flex; gap: 15px; justify-content: space-around;">
        """
        
        for i, (algo, prob) in enumerate(zip(['RF', 'SVM', 'XGB'], probabilities)):
            bar_color = '#c9533e' if prob >= 0.5 else '#495b8d'
            bar_height = prob * 100
            chart_html += f"""
                <div style="text-align: center;">
                    <div style="font-weight: bold; margin-bottom: 5px;">{algo}</div>
                    <div style="height: 120px; width: 40px; background: #e9ecef; border-radius: 4px; position: relative; margin: 0 auto;">
                        <div style="height: {bar_height}%; background: {bar_color}; width: 100%; border-radius: 4px; position: absolute; bottom: 0;"></div>
                    </div>
                    <div style="margin-top: 8px; font-weight: bold; color: {bar_color};">{prob*100:.1f}%</div>
                </div>
            """
        
        chart_html += """
            </div>
            <div style="margin-top: 15px; padding: 10px; background: white; border-left: 3px solid #495b8d; border-radius: 4px;">
                <span style="font-weight: bold;">Ensemble Verdict:</span>
        """
        
        if avg_prob >= 0.6:
            verdict = f"<span style='color: #c9533e;'>HIGH RISK — {consensus_label}</span>"
        elif avg_prob >= 0.4:
            verdict = f"<span style='color: #f59f00;'>MODERATE RISK — {consensus_label}</span>"
        else:
            verdict = f"<span style='color: #495b8d;'>LOWER RISK — {consensus_label}</span>"
        
        chart_html += verdict + "</div></div>"
    else:
        chart_html = "<p style='color: red;'>⚠️ No models loaded. Run training notebooks first.</p>"
    
    # Final verdict text
    if avg_prob >= 0.6:
        verdict_text = f"🚨 **HIGH RISK** — Ensemble average: {avg_prob*100:.1f}% fatal probability"
    elif avg_prob >= 0.4:
        verdict_text = f"⚠️ **MODERATE RISK** — Ensemble average: {avg_prob*100:.1f}% fatal probability"
    else:
        verdict_text = f"✓ **LOWER RISK** — Ensemble average: {avg_prob*100:.1f}% fatal probability"
    
    map_html = build_map(avg_prob)
    
    return predictions_df, chart_html, map_html, verdict_text

print('✓ Prediction & visualization function ready')

## 7. Gradio Dashboard Interface

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# ── Dataset summary (for the "About the Data" panel) ─────────────────────────
if 'combined_df' in locals() and TARGET_COL in combined_df.columns:
    n_total   = len(combined_df)
    n_fatal   = int(combined_df[TARGET_COL].sum())
    fatal_pct = n_fatal / n_total * 100
    dataset_summary_md = f"""
| Metric | Value |
|---|---|
| Total combined records | {n_total:,} |
| Fatal incidents | {n_fatal:,} ({fatal_pct:.1f}%) |
| Non-fatal incidents | {n_total - n_fatal:,} ({100-fatal_pct:.1f}%) |
| Train set size | {len(train_df) if 'train_df' in locals() else 'N/A'} |
| Test set size | {len(test_df) if 'test_df' in locals() else 'N/A'} |
"""
else:
    dataset_summary_md = "*Dataset summary unavailable — run `data_preparation.ipynb` first.*"

print(dataset_summary_md)

# ── Test set performance metrics for all 3 models (computed once) ───────────
def compute_test_metrics():
    if 'test_df' not in locals() and 'test_df' not in globals():
        return pd.DataFrame()

    X_test = test_df[FEATURE_COLS]
    y_test = test_df[TARGET_COL]
    rows = []
    for algo in ['RF', 'SVM', 'XGB']:
        if algo in models:
            try:
                y_pred = models[algo].predict(X_test)
                y_prob = models[algo].predict_proba(X_test)[:, 1]
                rows.append({
                    'Model': {'RF': 'Random Forest', 'SVM': 'SVM', 'XGB': 'XGBoost'}[algo],
                    'Accuracy': round(accuracy_score(y_test, y_pred), 4),
                    'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
                    'Recall': round(recall_score(y_test, y_pred, zero_division=0), 4),
                    'F1-Score': round(f1_score(y_test, y_pred, zero_division=0), 4),
                    'ROC-AUC': round(roc_auc_score(y_test, y_prob), 4),
                })
            except Exception as e:
                rows.append({'Model': algo, 'Accuracy': f'Error: {e}', 'Precision': '—',
                             'Recall': '—', 'F1-Score': '—', 'ROC-AUC': '—'})
    return pd.DataFrame(rows)

test_metrics_df = compute_test_metrics()
print('\nTest Set Performance (all models):')
print(test_metrics_df.to_string(index=False) if not test_metrics_df.empty else 'No models/test data loaded.')

## 6.1 Precompute Dataset Summary & Test Set Performance

These are computed once at startup so the dashboard can display them instantly
(no need to recompute on every prediction click).

In [ ]:
with gr.Blocks(
    title='Marine Incident Fatality Predictor',
    theme=gr.themes.Soft(primary_hue='teal', secondary_hue='blue'),
    css=".comparison_chart { margin: 20px 0; }"
) as demo:

    gr.Markdown("""
    # 🦈 Marine Incident Fatality Predictor
    **Cloud ML Project — Unified Dataset Approach (Combined Data)**
    
    Real-time fatality risk assessment using **Random Forest**, **SVM**, and **XGBoost**  
    all trained on a single unified dataset (34k+ combined records, 80/20 stratified split).
    
    Compare predictions across all 3 algorithms in real-time.
    """)

    with gr.Tabs():
        # ── Tab 1: Live Prediction ────────────────────────────────────────
        with gr.Tab('🔍 Live Prediction'):

            gr.Markdown('### ⚡ Quick Presets')
            with gr.Row():
                preset_surf_btn = gr.Button('🏄 Surfing incident')
                preset_dive_btn = gr.Button('🤿 Diving incident')
                preset_swim_btn = gr.Button('🏊 Swimming incident')
                reset_btn       = gr.Button('↺ Reset to defaults')

            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown('### 📋 Incident Details')

                    year = gr.Slider(minimum=1900, maximum=2026, value=2024, step=1,
                                    label='Year')
                    month = gr.Dropdown(
                        choices=[('January',1),('February',2),('March',3),('April',4),
                                 ('May',5),('June',6),('July',7),('August',8),
                                 ('September',9),('October',10),('November',11),('December',12)],
                        value=6, label='Month')
                    age = gr.Slider(minimum=1, maximum=100, value=30, step=1,
                                   label='Age (years)')
                    gender = gr.Radio(choices=['M', 'F', 'Unknown'], value='M',
                                     label='Gender')
                    activity_label = gr.Dropdown(choices=ACTIVITY_CHOICES,
                                                value=ACTIVITY_CHOICES[0] if ACTIVITY_CHOICES else None,
                                                label='Activity')
                    inc_type_label = gr.Dropdown(choices=INC_TYPE_CHOICES,
                                                value=INC_TYPE_CHOICES[0] if INC_TYPE_CHOICES else None,
                                                label='Incident Type')

                    predict_btn = gr.Button('🔍 Predict Fatality Risk', variant='primary', size='lg')

                with gr.Column(scale=2):
                    gr.Markdown('### 📊 Ensemble Model Predictions')
                    verdict_box = gr.Textbox(label='Overall Risk Verdict', interactive=False,
                                            lines=2, scale=1)
                    results_tbl = gr.Dataframe(label='Individual Model Results (RF / SVM / XGBoost)',
                                              interactive=False, scale=1)
                    comparison_viz = gr.HTML(label='Probability Comparison Chart', scale=1)

            gr.Markdown('### 🗺️ Historical Incident Map')
            gr.Markdown('*Red = historical fatal | Blue = historical non-fatal | Coloured ring = ensemble prediction (at combined dataset median location)*')
            map_out = gr.HTML(label='Interactive Map')

            gr.Markdown('---')
            gr.Markdown("""
            ### 📌 About This Model
            - **Unified Dataset**: All 3 models trained on the same combined data (no per-dataset selection needed)
            - **Same Train/Test Split**: 80/20 stratified split for fair comparison
            - **Feature Vector**: 9 features (year, month, age, gender, activity, incident type, lat/lon, source)
            - **Ensemble Approach**: Compare all 3 algorithms on identical data
            - **Location handling**: Latitude/longitude/source are resolved automatically from
              combined dataset statistics — no manual entry needed
            - **Educational Purpose**: Model predictions are for learning; not a substitute for professional advice
            """)

        # ── Tab 2: Model Performance & Dataset Overview ──────────────────
        with gr.Tab('📈 Model Performance & Dataset'):
            gr.Markdown('### 📦 Combined Dataset Overview')
            gr.Markdown(dataset_summary_md)

            gr.Markdown('### 🏆 Test Set Performance (held-out 20% split)')
            gr.Dataframe(value=test_metrics_df, label='Accuracy / Precision / Recall / F1 / ROC-AUC per model',
                        interactive=False)
            gr.Markdown("""
            *All 3 models were evaluated on the exact same held-out test set
            (`combined_test.csv`) for a fair, apples-to-apples comparison.*
            """)

    # ── Preset & reset logic ──────────────────────────────────────────────
    def _first_match(choices, keyword, fallback):
        for c in choices:
            if keyword.lower() in str(c).lower():
                return c
        return fallback

    def load_preset_surf():
        act = _first_match(ACTIVITY_CHOICES, 'surf', ACTIVITY_CHOICES[0] if ACTIVITY_CHOICES else None)
        inc = _first_match(INC_TYPE_CHOICES, 'unprovoked', INC_TYPE_CHOICES[0] if INC_TYPE_CHOICES else None)
        return 2024, 7, 28, 'M', act, inc

    def load_preset_dive():
        act = _first_match(ACTIVITY_CHOICES, 'div', ACTIVITY_CHOICES[0] if ACTIVITY_CHOICES else None)
        inc = _first_match(INC_TYPE_CHOICES, 'provoked', INC_TYPE_CHOICES[0] if INC_TYPE_CHOICES else None)
        return 2023, 3, 35, 'M', act, inc

    def load_preset_swim():
        act = _first_match(ACTIVITY_CHOICES, 'swim', ACTIVITY_CHOICES[0] if ACTIVITY_CHOICES else None)
        inc = _first_match(INC_TYPE_CHOICES, 'unprovoked', INC_TYPE_CHOICES[0] if INC_TYPE_CHOICES else None)
        return 2022, 1, 22, 'F', act, inc

    def reset_defaults():
        return (2024, 6, 30, 'M',
               ACTIVITY_CHOICES[0] if ACTIVITY_CHOICES else None,
               INC_TYPE_CHOICES[0] if INC_TYPE_CHOICES else None)

    preset_outputs = [year, month, age, gender, activity_label, inc_type_label]
    preset_surf_btn.click(fn=load_preset_surf, inputs=[], outputs=preset_outputs)
    preset_dive_btn.click(fn=load_preset_dive, inputs=[], outputs=preset_outputs)
    preset_swim_btn.click(fn=load_preset_swim, inputs=[], outputs=preset_outputs)
    reset_btn.click(fn=reset_defaults, inputs=[], outputs=preset_outputs)

    # ── Prediction wiring ──────────────────────────────────────────────────
    predict_btn.click(
        fn=predict_and_visualise,
        inputs=[year, month, age, gender, activity_label, inc_type_label],
        outputs=[results_tbl, comparison_viz, map_out, verdict_box]
    )

# share=True creates a public URL (required for Colab)
demo.launch(share=True, debug=False)

## 8. Batch Prediction API (Cloud Service Demo)
Demonstrates the **batch inference** service pathway — upload a CSV, get predictions appended.

In [ ]:
import time

def batch_predict_on_test_set():
    """
    Batch inference on test dataset:
      - Loads combined_test.csv
      - Runs all 3 combined models
      - Returns predictions and comparison metrics
      - Shows sample predictions
    """
    if 'test_df' not in locals():
        return None, "Test dataset not loaded"
    
    print(f'Running batch inference on {len(test_df)} test samples...')
    
    # Extract features
    X_test = test_df[FEATURE_COLS].copy()
    y_true = test_df[TARGET_COL].copy()
    
    # Initialize prediction columns
    predictions_dict = {}
    probabilities_dict = {}
    
    t0 = time.perf_counter()
    
    # Run each model
    for algo in ['RF', 'SVM', 'XGB']:
        if algo in models:
            try:
                model = models[algo]
                y_pred = model.predict(X_test)
                y_prob = model.predict_proba(X_test)[:, 1]
                
                predictions_dict[algo] = y_pred
                probabilities_dict[algo] = y_prob
                print(f'✓ {algo}: {len(y_pred)} predictions')
            except Exception as e:
                print(f'✗ {algo}: Error - {str(e)}')
    
    elapsed = time.perf_counter() - t0
    throughput = len(test_df) / elapsed
    
    # Build results dataframe
    results_df = test_df[[col for col in [TARGET_COL] + FEATURE_COLS if col in test_df.columns]].copy()
    
    for algo, preds in predictions_dict.items():
        results_df[f'pred_{algo}'] = preds
        results_df[f'prob_{algo}'] = probabilities_dict[algo]
    
    # Calculate ensemble predictions
    if predictions_dict:
        ensemble_preds = np.round(np.mean([probabilities_dict[a] for a in predictions_dict.keys()], axis=0) >= 0.5).astype(int)
        results_df['ensemble_pred'] = ensemble_preds
        results_df['ensemble_prob'] = np.mean([probabilities_dict[a] for a in predictions_dict.keys()], axis=0)
    
    # Calculate metrics for each model
    metrics = {}
    for algo, preds in predictions_dict.items():
        from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
        try:
            acc = accuracy_score(y_true, preds)
            prec = precision_score(y_true, preds, zero_division=0)
            rec = recall_score(y_true, preds, zero_division=0)
            f1 = f1_score(y_true, preds, zero_division=0)
            auc = roc_auc_score(y_true, probabilities_dict[algo])
            
            metrics[algo] = {
                'Accuracy': f'{acc:.4f}',
                'Precision': f'{prec:.4f}',
                'Recall': f'{rec:.4f}',
                'F1-Score': f'{f1:.4f}',
                'ROC-AUC': f'{auc:.4f}'
            }
        except:
            pass
    
    # Create summary HTML
    summary_html = f"""
    <div style="padding: 20px; background: #f8f9fa; border-radius: 8px;">
        <h3>Batch Inference Results on Test Set</h3>
        <p><strong>Test samples:</strong> {len(test_df)} | <strong>Models:</strong> {len(predictions_dict)}/3 loaded</p>
        <p><strong>Inference time:</strong> {elapsed:.2f}s | <strong>Throughput:</strong> {throughput:.0f} samples/sec</p>
        
        <h4>Performance Metrics</h4>
        <table style="border-collapse: collapse; width: 100%;">
            <tr style="background: #e9ecef;">
                <th style="border: 1px solid #ddd; padding: 8px;">Model</th>
                <th style="border: 1px solid #ddd; padding: 8px;">Accuracy</th>
                <th style="border: 1px solid #ddd; padding: 8px;">Precision</th>
                <th style="border: 1px solid #ddd; padding: 8px;">Recall</th>
                <th style="border: 1px solid #ddd; padding: 8px;">F1-Score</th>
                <th style="border: 1px solid #ddd; padding: 8px;">ROC-AUC</th>
            </tr>
    """
    
    for algo, met in metrics.items():
        summary_html += f"""
            <tr>
                <td style="border: 1px solid #ddd; padding: 8px; font-weight: bold;">{algo}</td>
                <td style="border: 1px solid #ddd; padding: 8px;">{met['Accuracy']}</td>
                <td style="border: 1px solid #ddd; padding: 8px;">{met['Precision']}</td>
                <td style="border: 1px solid #ddd; padding: 8px;">{met['Recall']}</td>
                <td style="border: 1px solid #ddd; padding: 8px;">{met['F1-Score']}</td>
                <td style="border: 1px solid #ddd; padding: 8px;">{met['ROC-AUC']}</td>
            </tr>
        """
    
    summary_html += """
        </table>
    </div>
    """
    
    return results_df, summary_html

# Run batch inference on test set
print('='*70)
test_results, test_summary = batch_predict_on_test_set()
print('='*70)

if test_summary:
    print(test_summary)

if test_results is not None:
    print(f'\nSample predictions (first 10):')
    pred_cols = [col for col in test_results.columns if 'pred' in col or 'prob' in col]
    print(test_results[[TARGET_COL] + FEATURE_COLS[:3] + pred_cols].head(10).to_string())
else:
    print('Test set inference could not be completed')

## 9. Dashboard Architecture Summary

**Unified Dataset Approach — 3 Combined Models**

```
┌─────────────────────────────────────────────────────────────────┐
│              Google Colab (Cloud Compute)                       │
│                                                                 │
│  ┌──────────────────┐    ┌────────────────────────────────┐    │
│  │  Google Drive    │───▶│  3 Combined Models (.pkl)      │    │
│  │  (Storage)       │    │  ✓ RF_combined.pkl            │    │
│  │  ├─ combined_*.csv    │  ✓ SVM_combined.pkl           │    │
│  │  ├─ models/      │    │  ✓ XGBoost_combined.pkl       │    │
│  │  └─ datasets/    │    └───────────────┬────────────────┘    │
│  └──────────────────┘                    │                     │
│                                          │                     │
│  ┌──────────────────────────────────────▼──────────────────┐   │
│  │   Gradio Web Server (dashboard.ipynb)                   │   │
│  │                                                         │   │
│  │  Input Form (year, month, age, gender, ...) ──┐        │   │
│  │                                               │        │   │
│  │  ┌─────────────────────────────────────────┐ │        │   │
│  │  │  Ensemble Prediction Engine             │ │        │   │
│  │  │  ├─ RF Model    → [0.65 fatal prob]    │ │        │   │
│  │  │  ├─ SVM Model   → [0.72 fatal prob]    │ │        │   │
│  │  │  └─ XGB Model   → [0.68 fatal prob]    │ │        │   │
│  │  │                                         │ │        │   │
│  │  │  ENSEMBLE: 68% avg → **HIGH RISK** ⚠️   │ │        │   │
│  │  └─────────────────────────────────────────┘ │        │   │
│  │                                               │        │   │
│  │  Output: Visual Comparison (bar chart) ◄─────┘        │   │
│  │          Probability table (all 3 models)             │   │
│  │          Risk verdict & confidence                    │   │
│  └──────────────────────────────────┬──────────────────┘   │
│                                     │ share=True            │
│                                     │ (Public HTTPS URL)    │
└─────────────────────────────────────┼───────────────────────┘
                                      │
                            ┌─────────▼──────────┐
                            │   End Users (Web)  │
                            │  • Form inputs     │
                            │  • Visual compare  │
                            │  • Risk alerts     │
                            └────────────────────┘
```

**Services provided:**
- **Real-time ensemble inference** — Input form → instant predictions from all 3 models with visual comparison
- **Unified dataset models** — All trained on same 80/20 stratified split (34k+ combined records)
- **Batch test set inference** — Load test.csv, run all models, display performance metrics (Accuracy, Precision, Recall, F1, AUC)
- **Visual comparison** — Bar chart showing each model's fatality probability, consensus verdict
- **Public URL** — Gradio `share=True` creates a 72-hour shareable public endpoint

**Cloud technologies used:**
- Google Colab (serverless compute)
- Google Drive (unified dataset + combined models storage)
- Gradio (web API + interactive UI)
- scikit-learn & XGBoost (model inference)
- pandas & numpy (data processing)

---
# 10. AWS Production Pipeline

Running this project on **AWS** replaces each Google service with a managed AWS equivalent and adds enterprise features: auto-scaling, IAM access control, CloudWatch monitoring, and a CI/CD retraining trigger.

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                              AWS Cloud                                      │
│                                                                             │
│  ① DATA LAYER                                                               │
│  ┌─────────────────────┐    ┌────────────────────────┐                     │
│  │  Amazon S3          │    │  AWS Glue (ETL)        │                     │
│  │  s3://cloudml-data/ │───▶│  Clean & Transform CSVs│                     │
│  │  ├─ attacks.csv     │    │  → Parquet on S3       │                     │
│  │  ├─ geocoded.csv    │    └────────────────────────┘                     │
│  │  └─ surf_zone.csv   │                                                   │
│  └─────────────────────┘                                                   │
│                                                                             │
│  ② TRAINING LAYER                                                           │
│  ┌─────────────────────────────────────────────────────────────────────┐   │
│  │  Amazon SageMaker Studio (Notebooks — equivalent to Colab)          │   │
│  │  ┌──────────────────────────────────────────────────────────────┐   │   │
│  │  │  SageMaker Training Jobs (managed, parallelisable)           │   │   │
│  │  │  teammate1_random_forest  │  teammate2_svm  │ teammate3_xgb  │   │   │
│  │  └──────────────────────────────────────────────────────────────┘   │   │
│  │  → Trained .pkl artifacts saved to  s3://cloudml-data/models/       │   │
│  └─────────────────────────────────────────────────────────────────────┘   │
│                                                                             │
│  ③ MODEL REGISTRY                                                           │
│  ┌─────────────────────────────────┐                                        │
│  │  SageMaker Model Registry       │  ← versioned, auditable               │
│  └─────────────────────────────────┘                                        │
│                                                                             │
│  ④ SERVING LAYER  (two options — choose based on traffic)                   │
│                                                                             │
│  Option A — Real-time (high traffic)                                        │
│  ┌──────────────────────────────────────────────────────────────────────┐  │
│  │  SageMaker Endpoints  (auto-scaling, multi-model capable)            │  │
│  │  endpoint: rf-fatality-predictor                                     │  │
│  │  endpoint: svm-fatality-predictor                                    │  │
│  │  endpoint: xgb-fatality-predictor                                    │  │
│  └──────────────────────────────────────────────────────────────────────┘  │
│                                                                             │
│  Option B — Serverless (low traffic / cost-optimised)                       │
│  ┌──────────────────────────────────────────────────────────────────────┐  │
│  │  AWS Lambda  (predict_handler.py)                                    │  │
│  │  → Loads model from S3 on cold start                                 │  │
│  │  → Returns JSON prediction in < 200 ms                               │  │
│  └──────────────────────────────────────────────────────────────────────┘  │
│          ▲                                                                  │
│  ┌───────┴──────────────┐                                                   │
│  │  Amazon API Gateway  │  ← REST API, auth via API Key / Cognito           │
│  └───────────────────────┘                                                  │
│                                                                             │
│  ⑤ DASHBOARD / FRONTEND                                                     │
│  ┌──────────────────────────────────────────────────────────────────────┐  │
│  │  AWS App Runner  ←  Docker image (Gradio + Folium app)               │  │
│  │  → Auto-scales to zero, HTTPS, custom domain                         │  │
│  └──────────────────────────────────────────────────────────────────────┘  │
│                                                                             │
│  ⑥ MONITORING                                                               │
│  ┌────────────────────────────────────┐                                     │
│  │  Amazon CloudWatch                 │  Latency, error rate, throughput    │
│  │  SageMaker Model Monitor           │  Data drift detection               │
│  │  AWS EventBridge + CodePipeline    │  Scheduled retraining trigger       │
│  └────────────────────────────────────┘                                     │
└─────────────────────────────────────────────────────────────────────────────┘
           │  Public HTTPS
   ┌───────▼───────────┐
   │   End Users       │  (browser / mobile)
   └───────────────────┘
```

| Component | Google Colab equivalent | AWS service |
|---|---|---|
| Compute (notebooks) | Google Colab | **SageMaker Studio** |
| Storage (data + models) | Google Drive | **Amazon S3** |
| ETL | pandas locally | **AWS Glue** |
| Training at scale | Sequential cells | **SageMaker Training Jobs** |
| Real-time serving | `demo.launch(share=True)` | **SageMaker Endpoints** |
| Serverless inference | — | **Lambda + API Gateway** |
| Dashboard hosting | Gradio public URL (72h) | **AWS App Runner** |
| Monitoring | — | **CloudWatch + Model Monitor** |
| CI/CD retraining | Manual re-run | **CodePipeline + EventBridge** |


### 10.1  S3 — Data & Model Storage

In [ ]:
"""
AWS S3 — Upload datasets and save/load trained models.

Prerequisites:
    pip install boto3
    aws configure  (or set env vars AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY)
"""

import boto3, joblib, io, os

# ── Configuration ────────────────────────────────────────────────────────────
AWS_REGION  = 'eu-west-1'           # change to your region
S3_BUCKET   = 'cloudml-shark-data'  # must exist: aws s3 mb s3://cloudml-shark-data
DATA_PREFIX = 'datasets/'
MODEL_PREFIX= 'models/'

s3 = boto3.client('s3', region_name=AWS_REGION)

# ── Upload datasets to S3 ────────────────────────────────────────────────────
def upload_datasets_to_s3(local_dir: str = './'):
    files = [
        'attacks.csv',
        'geocoded-global-shark-attacks.csv',
        'Surf_Zone_Fatalities_2010-Present.csv',
    ]
    for fname in files:
        local_path = os.path.join(local_dir, fname)
        s3_key     = DATA_PREFIX + fname
        s3.upload_file(local_path, S3_BUCKET, s3_key)
        print(f'Uploaded  s3://{S3_BUCKET}/{s3_key}')

# ── Save a trained model to S3 (replaces joblib.dump to local Drive) ─────────
def save_model_to_s3(model, model_name: str):
    """Serialise model in-memory and push directly to S3 (no local file)."""
    buf = io.BytesIO()
    joblib.dump(model, buf)
    buf.seek(0)
    s3_key = MODEL_PREFIX + model_name + '.pkl'
    s3.put_object(Bucket=S3_BUCKET, Key=s3_key, Body=buf.read())
    print(f'Model saved → s3://{S3_BUCKET}/{s3_key}')

# ── Load a model from S3 ─────────────────────────────────────────────────────
def load_model_from_s3(model_name: str):
    """Download and deserialise a model from S3 into memory."""
    s3_key  = MODEL_PREFIX + model_name + '.pkl'
    obj     = s3.get_object(Bucket=S3_BUCKET, Key=s3_key)
    buf     = io.BytesIO(obj['Body'].read())
    model   = joblib.load(buf)
    print(f'Model loaded ← s3://{S3_BUCKET}/{s3_key}')
    return model

# ── Read a dataset directly from S3 into pandas ──────────────────────────────
def read_csv_from_s3(filename: str) -> 'pd.DataFrame':
    import pandas as pd
    s3_key = DATA_PREFIX + filename
    obj    = s3.get_object(Bucket=S3_BUCKET, Key=s3_key)
    return pd.read_csv(io.BytesIO(obj['Body'].read()), encoding='latin-1', low_memory=False)

# ── Demo (comment out if AWS credentials are not yet configured) ──────────────
# upload_datasets_to_s3()
# save_model_to_s3(pipe1, 'random_forest_ds1')
# rf_ds1 = load_model_from_s3('random_forest_ds1')
# geo_df = read_csv_from_s3('geocoded-global-shark-attacks.csv')
print('S3 helper functions defined.')
print(f'Target bucket : s3://{S3_BUCKET}')


### 10.2  SageMaker Training Job — Train at Scale in the Cloud

In [ ]:
"""
SageMaker SKLearn Estimator — submits a managed training job.
The training script (train_rf.py) is the same code from teammate1_random_forest.ipynb,
refactored into a standalone Python file.

Prerequisites:  pip install sagemaker
"""

import sagemaker
from sagemaker.sklearn.estimator import SKLearn

# ── IAM Role ─────────────────────────────────────────────────────────────────
# In SageMaker Studio this is auto-resolved.
# On a local machine: create a role with AmazonSageMakerFullAccess policy.
ROLE = sagemaker.get_execution_role()   # or paste your role ARN string

# ── S3 input paths ────────────────────────────────────────────────────────────
INPUT_S3 = f's3://{S3_BUCKET}/{DATA_PREFIX}'   # training data location

# ── Training script (train_rf.py) — stored alongside this notebook ────────────
TRAIN_SCRIPT = 'train_rf.py'   # see cell below for contents

# ── Estimator ─────────────────────────────────────────────────────────────────
rf_estimator = SKLearn(
    entry_point        = TRAIN_SCRIPT,
    role               = ROLE,
    instance_type      = 'ml.m5.xlarge',   # 4 vCPUs, 16 GB RAM
    framework_version  = '1.2-1',
    py_version         = 'py3',
    output_path        = f's3://{S3_BUCKET}/{MODEL_PREFIX}',
    hyperparameters    = {
        'n-estimators': 200,
        'max-depth'   : 12,
        'dataset'     : 'ds1',             # pass ds1 / ds2 / ds3
    },
    base_job_name      = 'cloudml-rf-training',
)

# ── Launch training job (async — returns immediately) ─────────────────────────
# rf_estimator.fit({'train': INPUT_S3})   # uncomment to submit job
print('SageMaker SKLearn estimator configured.')
print(f'Instance  : ml.m5.xlarge')
print(f'Output    : s3://{S3_BUCKET}/{MODEL_PREFIX}')
print('Call rf_estimator.fit() to submit the training job.')


In [ ]:
%%writefile train_rf.py
"""
SageMaker training entry-point script.
Equivalent to the training cells in teammate1_random_forest.ipynb.
SageMaker injects:
  --model-dir  /opt/ml/model   (where to save the model)
  --train      /opt/ml/input/data/train   (where CSV data lands)
"""
import argparse, os, joblib
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, roc_auc_score, classification_report

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--n-estimators', type=int,   default=200)
    parser.add_argument('--max-depth',    type=int,   default=12)
    parser.add_argument('--dataset',      type=str,   default='ds1')
    parser.add_argument('--model-dir',    type=str,   default=os.environ.get('SM_MODEL_DIR', '.'))
    parser.add_argument('--train',        type=str,   default=os.environ.get('SM_CHANNEL_TRAIN', '.'))
    args = parser.parse_args()

    # Load data
    file_map = {
        'ds1': 'attacks.csv',
        'ds2': 'geocoded-global-shark-attacks.csv',
        'ds3': 'Surf_Zone_Fatalities_2010-Present.csv',
    }
    fpath = os.path.join(args.train, file_map[args.dataset])
    df = pd.read_csv(fpath, encoding='latin-1', low_memory=False)

    if args.dataset in ('ds1', 'ds2'):
        target_col = 'Fatal (Y/N)' if args.dataset == 'ds1' else 'fatal_y_n'
        df['fatal'] = df[target_col].str.strip().str.upper().map({'Y': 1, 'N': 0})
        df = df.dropna(subset=['fatal'])
        df = df[df['fatal'].isin([0, 1])]

        feat_cols = (['Year','Type','Country','Activity','Sex ','Age']
                     if args.dataset == 'ds1'
                     else ['year','type','country','activity','sex','age',
                           'NEW_Latitude_Location_Area_Country',
                           'NEW_Longitude_Location_Area_Country'])
    else:
        df['fatal'] = (df['Rescue Fatality'].str.strip().str.lower() == 'fatality').astype(int)
        feat_cols = ['Month','Day','Year','State','Hazard','Hazard_Subcat',
                     'Gender','Age','Tropical Related','Lat','Long']

    d = df[feat_cols + ['fatal']].copy()
    cat_cols = d.select_dtypes('object').columns.tolist()
    for col in cat_cols:
        d[col] = LabelEncoder().fit_transform(d[col].astype(str).str.strip())
    for col in d.columns:
        d[col] = pd.to_numeric(d[col], errors='coerce')

    X = d.drop('fatal', axis=1)
    y = d['fatal']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                         random_state=42, stratify=y)

    neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
    pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('clf', RandomForestClassifier(n_estimators=args.n_estimators,
                                       max_depth=args.max_depth,
                                       class_weight='balanced',
                                       random_state=42, n_jobs=-1))
    ])
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    print(classification_report(y_test, y_pred))
    print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
    print(f'F1     : {f1_score(y_test, y_pred, zero_division=0):.4f}')

    # SageMaker expects model saved to SM_MODEL_DIR
    out_path = os.path.join(args.model_dir, f'rf_{args.dataset}.pkl')
    joblib.dump(pipe, out_path)
    print(f'Model saved to {out_path}')

if __name__ == '__main__':
    main()


### 10.3  SageMaker Endpoint — Real-Time Inference (Option A)

In [ ]:
import json, time
import numpy as np

# ── Deploy model as a SageMaker endpoint ─────────────────────────────────────
# (Run AFTER rf_estimator.fit() completes)

# rf_predictor = rf_estimator.deploy(
#     initial_instance_count = 1,
#     instance_type          = 'ml.t3.medium',    # cheap, suitable for demo
#     endpoint_name          = 'rf-fatality-predictor',
# )
# print('Endpoint deployed:', rf_predictor.endpoint_name)

# ── Call an already-deployed endpoint via boto3 (no sagemaker SDK needed) ────
sm_runtime = boto3.client('sagemaker-runtime', region_name=AWS_REGION)

def invoke_sagemaker_endpoint(endpoint_name: str, features: list) -> dict:
    """
    Call a deployed SageMaker endpoint.
    `features` is a flat list of numeric values matching the training feature order.
    Returns a dict with 'prediction' (0/1) and 'probability' (float).
    """
    payload = json.dumps({'instances': [features]})
    t0      = time.perf_counter()

    response = sm_runtime.invoke_endpoint(
        EndpointName = endpoint_name,
        ContentType  = 'application/json',
        Body         = payload,
    )

    latency_ms = (time.perf_counter() - t0) * 1000
    result     = json.loads(response['Body'].read())
    return {
        'prediction'  : int(result['predictions'][0]),
        'probability' : float(result.get('probabilities', [[0, 0]])[0][1]),
        'latency_ms'  : round(latency_ms, 2),
    }

# ── Updated predict_and_visualise using SageMaker endpoints ──────────────────
ENDPOINTS = {
    'Random Forest' : 'rf-fatality-predictor',
    'SVM'           : 'svm-fatality-predictor',
    'XGBoost'       : 'xgb-fatality-predictor',
}

def predict_via_aws(activity, age, country, attack_type, gender):
    """
    Production version of predict_and_visualise —
    calls SageMaker endpoints instead of local models.
    """
    feat_row = build_feature_row(activity, age, country, attack_type, gender)
    features = feat_row[0].tolist()   # flatten to list for JSON serialisation

    rows = []
    avg_prob = 0.0
    for model_name, endpoint in ENDPOINTS.items():
        try:
            result = invoke_sagemaker_endpoint(endpoint, features)
            prob   = result['probability']
            avg_prob += prob
            rows.append({
                'Model'            : model_name,
                'Prediction'       : 'FATAL' if prob >= 0.5 else 'Non-Fatal',
                'Fatal Probability': f"{prob*100:.1f}%",
                'Latency (ms)'     : result['latency_ms'],
            })
        except Exception as e:
            rows.append({'Model': model_name, 'Prediction': f'Error: {e}',
                         'Fatal Probability': '—', 'Latency (ms)': '—'})

    avg_prob /= max(len(rows), 1)
    verdict   = ('⚠️  HIGH RISK — FATAL predicted' if avg_prob >= 0.5
                 else '✅  Non-Fatal predicted')
    map_html  = build_map(country, avg_prob)
    return rows, map_html, verdict

print('SageMaker endpoint helpers defined.')
print('Endpoint names:', list(ENDPOINTS.values()))
print('Call predict_via_aws() once endpoints are deployed.')


### 10.4  AWS Lambda — Serverless Inference (Option B, cost-optimised)

In [ ]:
%%writefile lambda_handler.py
"""
AWS Lambda function — predict_handler.py
Deploy this file (+ joblib, scikit-learn, numpy in a Lambda Layer) via:
    zip function.zip lambda_handler.py
    aws lambda create-function --function-name cloudml-predictor ...

API Gateway triggers this handler with a JSON POST body:
    {
      "activity": "Surfing", "age": 28, "country": "USA",
      "attack_type": "Unprovoked", "gender": "M"
    }
"""
import os, io, json, boto3, joblib
import numpy as np
from sklearn.preprocessing import LabelEncoder

S3_BUCKET    = os.environ.get('S3_BUCKET',    'cloudml-shark-data')
MODEL_PREFIX = os.environ.get('MODEL_PREFIX',  'models/')
AWS_REGION   = os.environ.get('AWS_REGION',    'eu-west-1')

# ── Cold-start: load all 3 DS2 models from S3 once per container ─────────────
_models = {}

def _load_models():
    s3 = boto3.client('s3', region_name=AWS_REGION)
    for algo, fname in [('RF',  'random_forest_ds2.pkl'),
                        ('SVM', 'svm_ds2.pkl'),
                        ('XGB', 'xgboost_ds2.pkl')]:
        key = MODEL_PREFIX + fname
        try:
            obj  = s3.get_object(Bucket=S3_BUCKET, Key=key)
            buf  = io.BytesIO(obj['Body'].read())
            _models[algo] = joblib.load(buf)
            print(f'Loaded {fname}')
        except Exception as e:
            print(f'WARN: could not load {fname}: {e}')

_load_models()   # executes once at cold start

# Simple ordinal encoders (must match training encoding)
_ACTIVITIES  = ['Surfing','Swimming','Diving','Fishing','Snorkeling',
                 'Wading','Body Surfing','Kayaking','Paddling','Unknown']
_COUNTRIES   = ['USA','AUSTRALIA','SOUTH AFRICA','BRAZIL','REUNION','Unknown']
_TYPES       = ['Unprovoked','Provoked','Boat','Sea Disaster','Unknown']
_GENDERS     = ['M','F','Unknown']

def _encode(lst, val):
    try:    return lst.index(str(val).strip())
    except: return 0

def _build_features(body: dict) -> np.ndarray:
    return np.array([[
        int(body.get('year',    2024)),
        _encode(_TYPES,      body.get('attack_type', 'Unprovoked')),
        _encode(_COUNTRIES,  body.get('country',     'Unknown')),
        _encode(_ACTIVITIES, body.get('activity',    'Surfing')),
        _encode(_GENDERS,    body.get('gender',      'M')),
        float(body.get('age', 30)),
        float(body.get('latitude',  0.0)),
        float(body.get('longitude', 0.0)),
    ]])

def lambda_handler(event, context):
    try:
        # API Gateway passes body as a string
        body = json.loads(event.get('body', '{}')) if isinstance(
            event.get('body'), str) else event

        features = _build_features(body)
        results  = {}
        for algo, model in _models.items():
            try:
                prob = float(model.predict_proba(features)[0, 1])
                results[algo] = {
                    'probability': round(prob, 4),
                    'prediction' : 'FATAL' if prob >= 0.5 else 'Non-Fatal',
                }
            except Exception as e:
                results[algo] = {'error': str(e)}

        avg_prob = np.mean([v['probability'] for v in results.values()
                            if 'probability' in v])
        payload  = {
            'models'          : results,
            'avg_probability' : round(float(avg_prob), 4),
            'overall_verdict' : 'FATAL' if avg_prob >= 0.5 else 'Non-Fatal',
        }
        return {
            'statusCode': 200,
            'headers'   : {'Content-Type': 'application/json',
                           'Access-Control-Allow-Origin': '*'},
            'body'      : json.dumps(payload),
        }

    except Exception as e:
        return {'statusCode': 500, 'body': json.dumps({'error': str(e)})}


### 10.5  CloudWatch — Latency & Throughput Monitoring

In [ ]:
"""
CloudWatch — push custom latency metrics for each model inference.
This runs alongside predict_via_aws() to record performance data.
"""
import time, datetime

cw = boto3.client('cloudwatch', region_name=AWS_REGION)

def push_latency_metric(model_name: str, latency_ms: float, dataset: str = 'ds2'):
    """Push a single inference latency reading to CloudWatch."""
    cw.put_metric_data(
        Namespace  = 'CloudML/SharkFatality',
        MetricData = [
            {
                'MetricName' : 'InferenceLatency',
                'Dimensions' : [
                    {'Name': 'Model',   'Value': model_name},
                    {'Name': 'Dataset', 'Value': dataset},
                ],
                'Timestamp'  : datetime.datetime.utcnow(),
                'Value'      : latency_ms,
                'Unit'       : 'Milliseconds',
            }
        ]
    )

def pull_latency_stats(model_name: str, period_minutes: int = 60) -> dict:
    """Retrieve average and p99 latency from CloudWatch for the last N minutes."""
    end   = datetime.datetime.utcnow()
    start = end - datetime.timedelta(minutes=period_minutes)

    def _get(stat):
        resp = cw.get_metric_statistics(
            Namespace  = 'CloudML/SharkFatality',
            MetricName = 'InferenceLatency',
            Dimensions = [{'Name': 'Model', 'Value': model_name}],
            StartTime  = start,
            EndTime    = end,
            Period     = period_minutes * 60,
            Statistics = [stat],
        )
        pts = resp.get('Datapoints', [])
        return round(pts[0][stat], 2) if pts else None

    return {
        'model'         : model_name,
        'avg_ms'        : _get('Average'),
        'max_ms'        : _get('Maximum'),
        'sample_count'  : _get('SampleCount'),
        'period_minutes': period_minutes,
    }

# ── SageMaker Model Monitor (data drift detection) ───────────────────────────
def create_model_monitor(endpoint_name: str, baseline_s3_uri: str):
    """
    Set up SageMaker Model Monitor to detect data drift.
    Baseline statistics are computed from the training dataset.
    Violations are reported to CloudWatch and can trigger SNS alerts.
    """
    from sagemaker.model_monitor import DefaultModelMonitor
    from sagemaker.model_monitor.dataset_format import DatasetFormat

    monitor = DefaultModelMonitor(
        role              = ROLE,
        instance_count    = 1,
        instance_type     = 'ml.m5.xlarge',
        volume_size_in_gb = 20,
    )
    monitor.suggest_baseline(
        baseline_dataset    = baseline_s3_uri,
        dataset_format      = DatasetFormat.csv(header=True),
        output_s3_uri       = f's3://{S3_BUCKET}/monitor/baseline/',
    )
    monitor.create_monitoring_schedule(
        monitor_schedule_name = f'{endpoint_name}-monitor',
        endpoint_input        = endpoint_name,
        output_s3_uri         = f's3://{S3_BUCKET}/monitor/reports/',
        statistics            = monitor.baseline_statistics(),
        constraints           = monitor.suggested_constraints(),
        schedule_cron_expression = 'cron(0 * ? * * *)',  # every hour
    )
    print(f'Model monitor created for: {endpoint_name}')
    return monitor

print('CloudWatch metric helpers defined.')
print('Call push_latency_metric() after each inference to log to CloudWatch.')
print('Call pull_latency_stats()  to retrieve summary stats for reporting.')


### 10.6  AWS CLI — Step-by-Step Deployment Commands

The shell commands below set up the full pipeline from scratch. Run them in order in a terminal with the **AWS CLI** configured (`aws configure`).

```bash
# ── 0. Prerequisites ─────────────────────────────────────────────────────────
pip install boto3 sagemaker awscli

# ── 1. Create S3 bucket (one-time) ──────────────────────────────────────────
aws s3 mb s3://cloudml-shark-data --region eu-west-1

# ── 2. Upload datasets ───────────────────────────────────────────────────────
aws s3 cp attacks.csv                          s3://cloudml-shark-data/datasets/
aws s3 cp geocoded-global-shark-attacks.csv    s3://cloudml-shark-data/datasets/
aws s3 cp Surf_Zone_Fatalities_2010-Present.csv s3://cloudml-shark-data/datasets/

# ── 3. Create IAM role for SageMaker (one-time) ──────────────────────────────
# In AWS Console → IAM → Create Role → SageMaker → attach AmazonSageMakerFullAccess

# ── 4. Submit SageMaker training jobs (one per algo × dataset = 9 jobs) ──────
# (run from notebook: rf_estimator.fit() / svm_estimator.fit() / xgb_estimator.fit())

# ── 5. Deploy models as SageMaker endpoints ──────────────────────────────────
# (run from notebook: rf_estimator.deploy(...))

# ── 6. Create Lambda function ────────────────────────────────────────────────
zip lambda_pkg.zip lambda_handler.py
aws lambda create-function \
    --function-name cloudml-predictor \
    --runtime python3.11 \
    --role arn:aws:iam::123456789:role/lambda-cloudml-role \
    --handler lambda_handler.lambda_handler \
    --zip-file fileb://lambda_pkg.zip \
    --timeout 30 \
    --memory-size 512 \
    --environment Variables="{S3_BUCKET=cloudml-shark-data,AWS_REGION=eu-west-1}"

# ── 7. Create API Gateway REST API ───────────────────────────────────────────
aws apigateway create-rest-api --name cloudml-predict-api
# (connect POST /predict → Lambda via console or aws apigateway commands)

# ── 8. Deploy Gradio dashboard on AWS App Runner ─────────────────────────────
# Build Docker image
cat > Dockerfile << 'EOF'
FROM python:3.11-slim
WORKDIR /app
COPY dashboard.ipynb requirements.txt ./
RUN pip install gradio folium boto3 joblib scikit-learn pandas numpy
CMD ["python", "-c", "import subprocess; subprocess.run(['jupyter','nbconvert','--to','script','dashboard.ipynb','--stdout']); exec(open('dashboard.py').read())"]
EOF

aws ecr create-repository --repository-name cloudml-dashboard
# push image → ECR → create App Runner service pointing to ECR image

# ── 9. Enable CloudWatch monitoring ──────────────────────────────────────────
# Automatic for Lambda and SageMaker endpoints — no setup needed
# Custom metrics: call push_latency_metric() in your inference code

# ── 10. Set up EventBridge retraining trigger (weekly) ───────────────────────
aws events put-rule \
    --name cloudml-weekly-retrain \
    --schedule-expression "cron(0 2 ? * MON *)" \
    --state ENABLED
# target: SageMaker pipeline or Step Functions workflow
```

**Estimated Monthly Cost (AWS Free Tier + small production):**
| Service | Usage | Estimated cost |
|---|---|---|
| S3 (5 GB) | Data + models | ~$0.12/month |
| SageMaker Endpoint (ml.t3.medium) | 720 hrs | ~$43/month |
| Lambda (1M invocations) | Serverless | ~$0.20/month |
| App Runner (0.25 vCPU) | Dashboard | ~$5/month |
| CloudWatch | Basic metrics | Free tier |
| **Serverless-only total** | Lambda + S3 + App Runner | **~$5.32/month** |
| **SageMaker total** | Endpoint + S3 + App Runner | **~$48/month** |
